# SeeDL: Deep Learning Visualization & Observability Showcase

Welcome to **SeeDL** (`seedl`), a deep-learning-aware visualization and observability library for Python.

This notebook demonstrates the complete public API of SeeDL:
1. **Multidimensional Tensor Visualization** (`sd.see`)
2. **Model Architecture & Shape Tracing** (`sd.see(model, input=...)`)
3. **Training Observability & Run Comparison** (`sd.watch`, `sd.compare`)
4. **Layer Activations & Feature Maps** (`sd.activations`)
5. **Gradient Flow Observability** (`sd.gradients`)
6. **Prediction Analysis & Model Comparison** (`sd.predictions`, `sd.compare_predictions`)
7. **Transformer Attention & Convolution Simulation** (`sd.attention`, `sd.convolution`)
8. **Vector Embeddings & 2D PCA** (`sd.embeddings`)
9. **Explainability & Attribution** (`sd.explain`: Saliency, Integrated Gradients, Grad-CAM)
10. **Educational Simulations** (`sd.learn`)
11. **Multi-Backend & Export** (`.explore()`, `.save(...)`, `.to_figure(...)`)


In [1]:
import torch
from torch import nn

import seedl as sd

print("SeeDL version:", sd.__version__)
print("PyTorch version:", torch.__version__)

SeeDL version: 0.1.0
PyTorch version: 2.14.1+cu130


## 1. Multidimensional Tensor Visualization (`sd.see`)

SeeDL inspects scalars, 1D signals, 2D matrices, and high-dimensional tensors, formatting shapes, statistics, value ranges, and memory footprints automatically.


In [2]:
# 1.1 Scalar metric
sd.see(0.00345)

Visualization(kind='scalar', title='Numeric value', summary='float', data={'value': 0.00345, 'display_value': '0.00345', 'caption': 'float'}, metadata={'Type': 'float', 'Status': 'finite'}, notes=[], backend='plotly')

In [3]:
# 1.2 1D numeric sequence / signal
sd.see([1.2, 2.5, 3.1, 2.8, 4.0, 5.2, 4.9])

Visualization(kind='sequence', title='Numeric sequence', summary='7 value(s)', data={'values': [1.2, 2.5, 3.1, 2.8, 4.0, 5.2, 4.9], 'x': [0, 1, 2, 3, 4, 5, 6], 'label': 'Value'}, metadata={}, notes=[], backend='plotly')

In [4]:
# 1.3 2D matrix (correlation heatmap)
corr_matrix = [
    [1.0, 0.82, 0.15, -0.42],
    [0.82, 1.0, 0.05, -0.31],
    [0.15, 0.05, 1.0, 0.64],
    [-0.42, -0.31, 0.64, 1.0],
]
sd.see(corr_matrix)

Visualization(kind='matrix', title='Numeric matrix', summary='Matrix with shape (4, 4)', data={'values': [[1.0, 0.82, 0.15, -0.42], [0.82, 1.0, 0.05, -0.31], [0.15, 0.05, 1.0, 0.64], [-0.42, -0.31, 0.64, 1.0]], 'shape': (4, 4), 'row_indices': [0, 1, 2, 3], 'column_indices': [0, 1, 2, 3]}, metadata={}, notes=[], backend='plotly')

In [5]:
# 1.4 High-dimensional image tensor (CHW normalized RGB image)
img_tensor = torch.zeros(3, 32, 32)
img_tensor[0, :16, :] = 1.0  # Red upper half
img_tensor[1, :, :16] = 0.8  # Green left half
img_tensor[2, 16:, 16:] = 0.9  # Blue lower-right quadrant
sd.see(img_tensor)

Visualization(kind='image', title='Tensor visualization', summary='Image tensor with shape (3, 32, 32)', data={'pixels': [[[1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0]], [[1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000011920929, 0.0], [1.0, 0.800000

## 2. Model Architecture & Shape Tracing (`sd.see(model)`)

Inspect neural network hierarchies, parameter counts (trainable vs. frozen), shared parameter deduplication, and hook-driven tensor shape propagation.


In [6]:
# Define a convolutional network
class ConvClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(16 * 4 * 4, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        h = self.relu(self.bn1(self.conv1(x)))
        p = self.flatten(self.pool(h))
        return self.fc2(torch.relu(self.fc1(p)))


model = ConvClassifier()
sample_input = torch.randn(2, 3, 32, 32)

# Inspect model hierarchy and parameter counts:
sd.see(model)

Visualization(kind='model', title='ConvClassifier architecture', summary='8 module(s) · 9,034 parameter(s) · 9,034 trainable', data={'layers': [{'name': '', 'depth': 0, 'type': 'ConvClassifier', 'parameters': 0, 'trainable': 0, 'input_shape': '—', 'output_shape': '—'}, {'name': 'conv1', 'depth': 1, 'type': 'Conv2d', 'parameters': 448, 'trainable': 448, 'input_shape': '—', 'output_shape': '—'}, {'name': 'bn1', 'depth': 1, 'type': 'BatchNorm2d', 'parameters': 32, 'trainable': 32, 'input_shape': '—', 'output_shape': '—'}, {'name': 'relu', 'depth': 1, 'type': 'ReLU', 'parameters': 0, 'trainable': 0, 'input_shape': '—', 'output_shape': '—'}, {'name': 'pool', 'depth': 1, 'type': 'AdaptiveAvgPool2d', 'parameters': 0, 'trainable': 0, 'input_shape': '—', 'output_shape': '—'}, {'name': 'flatten', 'depth': 1, 'type': 'Flatten', 'parameters': 0, 'trainable': 0, 'input_shape': '—', 'output_shape': '—'}, {'name': 'fc1', 'depth': 1, 'type': 'Linear', 'parameters': 8224, 'trainable': 8224, 'input_shap

In [7]:
# Trace input and output shapes across every layer:
sd.see(model, input=sample_input)

Visualization(kind='model', title='ConvClassifier architecture', summary='8 module(s) · 9,034 parameter(s) · 9,034 trainable', data={'layers': [{'name': '', 'depth': 0, 'type': 'ConvClassifier', 'parameters': 0, 'trainable': 0, 'input_shape': '((2, 3, 32, 32),)', 'output_shape': '(2, 10)'}, {'name': 'conv1', 'depth': 1, 'type': 'Conv2d', 'parameters': 448, 'trainable': 448, 'input_shape': '((2, 3, 32, 32),)', 'output_shape': '(2, 16, 32, 32)'}, {'name': 'bn1', 'depth': 1, 'type': 'BatchNorm2d', 'parameters': 32, 'trainable': 32, 'input_shape': '((2, 16, 32, 32),)', 'output_shape': '(2, 16, 32, 32)'}, {'name': 'relu', 'depth': 1, 'type': 'ReLU', 'parameters': 0, 'trainable': 0, 'input_shape': '((2, 16, 32, 32),)', 'output_shape': '(2, 16, 32, 32)'}, {'name': 'pool', 'depth': 1, 'type': 'AdaptiveAvgPool2d', 'parameters': 0, 'trainable': 0, 'input_shape': '((2, 16, 32, 32),)', 'output_shape': '(2, 16, 4, 4)'}, {'name': 'flatten', 'depth': 1, 'type': 'Flatten', 'parameters': 0, 'trainable'

## 3. Training Observability & Run Comparison (`sd.watch`, `sd.compare`)

Track loss and metrics during training with the lightweight `Tracker`. Visualizations include exponential moving average (EMA) smoothing and plateau/divergence signals.


In [8]:
tracker = sd.watch(name="baseline_experiment")

# Simulate 8 training epochs
losses = [1.25, 0.92, 0.68, 0.51, 0.42, 0.38, 0.35, 0.33]
val_losses = [1.30, 0.98, 0.74, 0.60, 0.55, 0.54, 0.56, 0.59]
accuracies = [0.45, 0.62, 0.73, 0.81, 0.85, 0.88, 0.90, 0.91]

for epoch, (train_loss, val_loss, acc) in enumerate(
    zip(losses, val_losses, accuracies, strict=True), start=1
):
    tracker.log(epoch=epoch, loss=train_loss, val_loss=val_loss, accuracy=acc)

# Visualize training curves
sd.see(tracker)

Visualization(kind='training', title='Training history', summary='3 metric(s), 8–8 epochs', data={'groups': [{'name': 'loss', 'series': [{'name': 'loss', 'role': 'training', 'x': [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0], 'values': [1.25, 0.92, 0.68, 0.51, 0.42, 0.38, 0.35, 0.33]}, {'name': 'val_loss', 'role': 'validation', 'x': [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0], 'values': [1.3, 0.98, 0.74, 0.6, 0.55, 0.54, 0.56, 0.59]}]}, {'name': 'accuracy', 'series': [{'name': 'accuracy', 'role': 'training', 'x': [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0], 'values': [0.45, 0.62, 0.73, 0.81, 0.85, 0.88, 0.9, 0.91]}]}], 'metrics': {'loss': [1.25, 0.92, 0.68, 0.51, 0.42, 0.38, 0.35, 0.33], 'val_loss': [1.3, 0.98, 0.74, 0.6, 0.55, 0.54, 0.56, 0.59], 'accuracy': [0.45, 0.62, 0.73, 0.81, 0.85, 0.88, 0.9, 0.91]}, 'summary': '3 metric(s), 8–8 epochs'}, metadata={}, notes=['Curves are diagnostic clues; they are not guarantees about model quality.', 'Possible overfitting clue in loss: training loss fell w

In [9]:
# Compare two training runs side by side
run_sgd = {"epochs": [1, 2, 3, 4], "loss": [1.4, 1.1, 0.9, 0.75]}
run_adam = {"epochs": [1, 2, 3, 4], "loss": [1.2, 0.7, 0.4, 0.25]}
sd.compare(run_sgd, run_adam, names=("SGD", "AdamW"))

Visualization(kind='history_comparison', title='Comparison: SGD vs AdamW', summary='Comparison of SGD vs AdamW across 1 metric group(s)', data={'groups': [{'name': 'loss', 'series': [{'name': 'SGD: loss', 'role': 'training', 'run': 'SGD', 'x': [1.0, 2.0, 3.0, 4.0], 'values': [1.4, 1.1, 0.9, 0.75]}, {'name': 'AdamW: loss', 'role': 'training', 'run': 'AdamW', 'x': [1.0, 2.0, 3.0, 4.0], 'values': [1.2, 0.7, 0.4, 0.25]}]}], 'run_a': 'SGD', 'run_b': 'AdamW', 'differences': ['loss: 0.75 vs 0.25 (diff: -0.5)']}, metadata={}, notes=['Comparing SGD and AdamW. Solid lines: SGD; Dashed lines: AdamW (or role-specific).', 'Final metric differences: loss: 0.75 vs 0.25 (diff: -0.5)'], backend='plotly')

## 4. Layer Activations & Feature Maps (`sd.activations`)

Capture intermediate activation tensors from an input-driven forward pass. SeeDL analyzes activation distributions and checks for dead units in threshold activation layers.


In [10]:
act_view = sd.activations(model, input=sample_input)
act_view

Visualization(kind='activations', title='ConvClassifier activations', summary='Captured 8 module activation(s)', data={'layers': [{'name': '(root)', 'type': 'ConvClassifier', 'shape': '(2, 10)', 'sample': [-0.002564098685979843, -0.09748787432909012, 0.07974647730588913, -0.11969758570194244, 0.06164991855621338, 0.2020154595375061, -0.0045712110586464405, -0.04323868826031685, -0.04325186461210251, -0.0029339641332626343, -0.0131332166492939, -0.10574908554553986, 0.07775472849607468, -0.11292795091867447, 0.040202587842941284, 0.20672932267189026, -0.006794449407607317, -0.046158820390701294, -0.05428178608417511, -0.0020549744367599487], 'mean': '0.00066265', 'std': '0.088939', 'max_abs': '0.20673', 'finite': 'yes', 'status': 'available', 'dead_fraction': 0.0, 'saturation_note': None, 'feature_maps': None}, {'name': 'conv1', 'type': 'Conv2d', 'shape': '(2, 16, 32, 32)', 'sample': [-0.3168428838253021, 0.6073155999183655, -0.878640353679657, -0.32131657004356384, -0.06802219897508621

## 5. Gradient Flow Observability (`sd.gradients`)

Non-intrusively observe gradients already present on model parameters after `.backward()`. Automatically monitors gradient norms, vanishing (<1e-7), exploding (>100), or all-zero layers without altering model state.


In [11]:
# Compute gradients from a loss objective
model.train()
out = model(sample_input).sum()
out.backward()

# Inspect parameter gradients
grad_view = sd.gradients(model)
grad_view

Visualization(kind='gradients', title='ConvClassifier gradients', summary='Gradients available for 8 of 8 parameter tensor(s)', data={'layers': [{'name': 'conv1.weight', 'shape': '(16, 3, 3, 3)', 'status': 'available', 'mean': '-0.0015705', 'std': '0.022909', 'max_abs': 0.0917719230055809, 'norm': 0.4772700369358063, 'l2_norm': '0.47727', 'finite': 'yes', 'trainable': True, 'sample': [-0.008738015778362751, -0.0149756520986557, -0.02082621306180954, 0.02571479231119156, -0.010114951990544796, 0.023877736181020737, -0.02114497870206833, -0.0004215543158352375, -0.025914281606674194, 0.014007437974214554, 0.026087693870067596, 0.020954914391040802, 0.014024234376847744, 0.0068654632195830345, -0.029773961752653122, -0.0016908636316657066, -0.022876013070344925, -0.0037697898223996162, -0.0279046930372715, 0.014817619696259499, 0.005863814614713192, 0.01076306402683258, -0.0028933221474289894, 0.035913947969675064, 0.0039391303434967995, 0.00968114286661148, -0.012066361494362354, -0.0083

## 6. Prediction Analysis & Comparison (`sd.predictions`, `sd.compare_predictions`)

Analyze multiclass classification ranking, normalized probability distributions, and regression error residuals ($y_{pred} - y_{true}$, MAE, MSE, RMSE).


In [12]:
# Multiclass classification scores with label mapping
logits = [3.2, 1.1, -0.8, 0.4, -2.1]
class_names = ["airplane", "automobile", "bird", "cat", "deer"]
sd.predictions(logits, labels=class_names, task="classification")

Visualization(kind='classification_predictions', title='Prediction Analysis', summary="Top prediction: 'airplane' with 82.9% confidence", data={'ranked': [{'rank': 1, 'class': 'airplane', 'index': 0, 'probability': 0.8287931192096104, 'raw_score': 3.2}, {'rank': 2, 'class': 'automobile', 'index': 1, 'probability': 0.10149104513905674, 'raw_score': 1.1}, {'rank': 3, 'class': 'cat', 'index': 3, 'probability': 0.05039896148248614, 'raw_score': 0.4}, {'rank': 4, 'class': 'bird', 'index': 2, 'probability': 0.015179875484910843, 'raw_score': -0.8}, {'rank': 5, 'class': 'deer', 'index': 4, 'probability': 0.004136998683935801, 'raw_score': -2.1}], 'top_class': 'airplane', 'top_prob': 0.8287931192096104, 'all_probs': [0.8287931192096104, 0.10149104513905674, 0.015179875484910843, 0.05039896148248614, 0.004136998683935801], 'classes': ['airplane', 'automobile', 'bird', 'cat', 'deer'], 'was_logits': True, 'is_batch': False, 'batch_size': 1}, metadata={}, notes=['Raw inputs were identified as raw 

In [13]:
# Regression prediction analysis against ground truth
y_pred = [24.5, 31.2, 18.0, 42.1, 15.3]
y_true = [23.8, 33.0, 17.5, 40.2, 16.0]
sd.predictions(y_pred, targets=y_true, task="regression")

Visualization(kind='regression_predictions', title='Prediction Analysis', summary='Regression: 5 sample(s) · MAE: 1.12 · RMSE: 1.271', data={'predictions': [24.5, 31.2, 18.0, 42.1, 15.3], 'count': 5, 'pred_min': 15.3, 'pred_max': 42.1, 'pred_mean': 26.22, 'targets': [23.8, 33.0, 17.5, 40.2, 16.0], 'residuals': [0.6999999999999993, -1.8000000000000007, 0.5, 1.8999999999999986, -0.6999999999999993], 'mae': 1.1199999999999997, 'mse': 1.615999999999999, 'rmse': 1.2712198865656559}, metadata={}, notes=['Regression output: 5 prediction value(s) ranging from 15.3 to 42.1.', 'Residual analysis available: MAE = 1.12, RMSE = 1.271.'], backend='plotly')

In [14]:
# Model prediction agreement / delta comparison
model_a_preds = [0.85, 0.10, 0.05]
model_b_preds = [0.70, 0.25, 0.05]
sd.compare_predictions(
    model_a_preds, model_b_preds, labels=["cat", "dog", "bird"], names=("ResNet", "ViT")
)

Visualization(kind='prediction_comparison', title='Prediction Comparison: ResNet vs ViT', summary="ResNet and ViT agree on top class: 'cat' (85.0%) vs 'cat' (70.0%)", data={'classes': ['cat', 'dog', 'bird'], 'rows': [{'class': 'cat', 'score_a': 0.85, 'score_b': 0.7, 'delta': -0.15000000000000002}, {'class': 'dog', 'score_a': 0.1, 'score_b': 0.25, 'delta': 0.15}, {'class': 'bird', 'score_a': 0.05, 'score_b': 0.05, 'delta': 0.0}], 'name_a': 'ResNet', 'name_b': 'ViT', 'top_a': 'cat', 'top_b': 'cat', 'top_score_a': 0.85, 'top_score_b': 0.7, 'agreement': True}, metadata={}, notes=['Comparing predictions from ResNet and ViT.', 'Agreement on top prediction: Yes.'], backend='plotly')

## 7. Transformer Attention & Convolution Simulation (`sd.attention`, `sd.convolution`)

Visualize multi-head self-attention and cross-attention matrices with token alignment, and simulate exact 2D cross-correlations with kernel arithmetic.


In [15]:
# Multi-head attention matrix with token alignment
attention_weights = [
    [0.70, 0.20, 0.08, 0.02],
    [0.10, 0.65, 0.20, 0.05],
    [0.05, 0.15, 0.60, 0.20],
    [0.02, 0.08, 0.25, 0.65],
]
tokens = ["The", "neural", "network", "learns"]
sd.attention(attention_weights, tokens=tokens)

Visualization(kind='attention', title='Transformer Attention', summary='Transformer Attention: 4 tokens · Single head (1 total heads)', data={'matrix': [[0.7, 0.2, 0.08, 0.02], [0.1, 0.65, 0.2, 0.05], [0.05, 0.15, 0.6, 0.2], [0.02, 0.08, 0.25, 0.65]], 'tokens': ['The', 'neural', 'network', 'learns'], 'seq_len': 4, 'num_heads': 1, 'selected_head': 'Single head', 'active_head': 'Single head', 'is_causal': False, 'shape': (4, 4)}, metadata={}, notes=['Query-Key attention heatmap: Rows represent Queries (from), Columns represent Keys (to).', 'Values form a valid probability distribution (rows sum to ~1.0).', 'Sequence: The neural network learns'], backend='plotly')

In [16]:
# 2D Convolution simulation: Sobel vertical edge filter
sobel_kernel = [
    [-1.0, 0.0, 1.0],
    [-2.0, 0.0, 2.0],
    [-1.0, 0.0, 1.0],
]
test_image = [
    [0.0, 0.0, 1.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.0, 0.0, 0.0],
]
sd.convolution(image=test_image, kernel=sobel_kernel, stride=1, padding=1)

Visualization(kind='convolution_simulation', title='2D Convolution Simulation', summary='2D Convolution: Input (6×6) * Kernel (3×3) -> Feature Map (6×6) [stride=1, pad=1]', data={'input_shape': (6, 6), 'kernel_shape': (3, 3), 'output_shape': (6, 6), 'h_out': 6, 'w_out': 6, 'stride': 1, 'padding': 1, 'input_image': [[0.0, 0.0, 1.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 1.0, 0.0, 0.0]], 'kernel': [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]], 'feature_map': [[0.0, 3.0, 3.0, -3.0, -3.0, 0.0], [0.0, 4.0, 4.0, -4.0, -4.0, 0.0], [0.0, 4.0, 4.0, -4.0, -4.0, 0.0], [0.0, 4.0, 4.0, -4.0, -4.0, 0.0], [0.0, 4.0, 4.0, -4.0, -4.0, 0.0], [0.0, 3.0, 3.0, -3.0, -3.0, 0.0]], 'output': [[0.0, 3.0, 3.0, -3.0, -3.0, 0.0], [0.0, 4.0, 4.0, -4.0, -4.0, 0.0], [0.0, 4.0, 4.0, -4.0, -4.0, 0.0], [0.0, 4.0, 4.0, -4.0, -4.0, 0.0], [0.0, 4.0, 4.0, -4.0, -4.0, 0.0], [0.0, 3.0, 3.0, -3.0, 

## 8. Vector Embeddings & 2D PCA (`sd.embeddings`)

Explore vector embeddings through pairwise cosine similarities and analytical 2D PCA projections with explained variance ratio.


In [17]:
vectors = [
    [1.0, 0.2, 0.0, 0.1],
    [0.9, 0.3, 0.1, 0.0],
    [0.0, 0.1, 0.9, 0.8],
    [0.1, 0.0, 0.8, 0.9],
]
labels = ["doc_science_1", "doc_science_2", "doc_art_1", "doc_art_2"]
sd.embeddings(vectors, labels=labels)

Visualization(kind='embeddings', title='Vector Embeddings', summary='Embeddings: 4 vectors in 4 dimensions (showing 4 samples)', data={'num_vectors': 4, 'dim': 4, 'labels': ['doc_science_1', 'doc_science_2', 'doc_art_1', 'doc_art_2'], 'similarity_matrix': [[0.9999999999999998, 0.9821003986826508, 0.08076606470596866, 0.1534555229413404], [0.9821003986826508, 1.0, 0.10410800445643575, 0.14748633964661734], [0.08076606470596866, 0.10410800445643575, 0.9999999999999999, 0.9863013698630136], [0.1534555229413404, 0.14748633964661734, 0.9863013698630136, 0.9999999999999999]], 'coords_2d': [[-0.7541317502536499, 0.09372994857230312], [-0.7053057776685169, -0.10021856109228763], [0.7541317502536499, -0.09372994857230314], [0.7053057776685169, 0.1002185610922876]], 'pca_points': [[-0.7541317502536499, 0.09372994857230312], [-0.7053057776685169, -0.10021856109228763], [0.7541317502536499, -0.09372994857230314], [0.7053057776685169, 0.1002185610922876]], 'explained_variance': 1.0, 'pca_variance_r

## 9. Feature Attribution & Explainability (`sd.explain`)

Inspect which input features influenced a model's prediction using:
- **Vanilla Saliency:** Gradient sensitivity w.r.t input.
- **Integrated Gradients:** Path-integrated gradients along a straight line from baseline using the composite midpoint integration rule (satisfying the Completeness axiom: $\sum \text{Attr} \approx F(x) - F(x_0)$).
- **Grad-CAM:** Coarse spatial activation mapping for convolutional layers.


In [18]:
# 9.1 Saliency Map
sd.explain(model, input=sample_input, method="saliency")

Visualization(kind='attribution', title='ConvClassifier Attribution (saliency)', summary='Attribution (Vanilla Saliency (Absolute Magnitude)) for target class 15 (score: 0.2111)', data={'method': 'saliency', 'target_class': 15, 'predicted_score': 0.21114873886108398, 'attribution': [[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 

In [19]:
# 9.2 Integrated Gradients (completeness-satisfying composite midpoint rule)
sd.explain(model, input=sample_input, method="integrated_gradients", steps=20)

Visualization(kind='attribution', title='ConvClassifier Attribution (integrated_gradients)', summary='Attribution (Integrated Gradients (20 steps via composite midpoint rule)) for target class 15 (score: 0.2111)', data={'method': 'integrated_gradients', 'target_class': 15, 'predicted_score': 0.21114873886108398, 'attribution': [[[[0.0, -0.0, 0.0, -0.0, -0.0, 0.0, -0.0, 0.0, -0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.0, -0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.0, -0.0, 0.0, 0.0, -0.0, 0.0, 0.0, -0.0, 0.0, -0.0, -0.0], [0.0, 0.0, 0.0, -0.0, 0.0, 0.0, -0.0, 0.0, -0.0, 0.0, -0.0, -0.0, 0.0, 0.0, 0.0, -0.0, -0.0, 0.0, -0.0, 0.0, -0.0, -0.0, -0.0, 0.0, 0.0, -0.0, 0.0, 0.0, 0.0, 0.0, -0.0, -0.0], [-0.0, 0.0, 0.0, -0.0, 0.0, 0.0, -0.0, 0.0, 0.0, 0.0, -0.0, 0.0, -0.0, -0.0, 0.0, -0.0, 0.0, 0.0, -0.0, -0.0, 0.0, -0.0, 0.0, -0.0, -0.0, -0.0, -0.0, -0.0, 0.0, 0.0, 0.0, -0.0], [0.0, -0.0, 0.0, -0.0, 0.0, -0.0, 0.0, -0.0, -0.0, -0.0, 0.0, 0.0, -0.0, -0.0, 0.0, -0.0, -0.0, -0.0, -0.0, 0.0, 0.0, 0.0, 0.0, -0.0, 0.0, 

In [20]:
# 9.3 Grad-CAM for Convolutional Network
sd.explain(model, input=sample_input, method="grad_cam")

Visualization(kind='attribution', title='ConvClassifier Attribution (grad_cam)', summary='Attribution (Grad-CAM (Class Activation Mapping)) for target class 15 (score: 0.2111)', data={'method': 'grad_cam', 'target_class': 15, 'predicted_score': 0.21114873886108398, 'attribution': [[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0

## 10. Educational Concept Simulations (`sd.learn`)

Explore core deep learning mechanisms through interactive mathematical simulations.


In [21]:
# Gradient Descent on f(x) = x^2 with learning rate 0.2
sd.learn("gradient_descent", lr=0.2, steps=10)

Visualization(kind='educational_gradient_descent', title='Learning: Gradient Descent', summary='Gradient Descent Simulation (lr=0.2): w=3.50 -> 0.3758 across 10 steps (final loss: 7.0616e-02)', data={'concept': 'gradient_descent', 'lr': 0.2, 'steps': 10, 'trajectory': [{'step': 0, 'w': 3.5, 'loss': 6.125, 'grad': 3.5}, {'step': 1, 'w': 2.8, 'loss': 3.9199999999999995, 'grad': 2.8}, {'step': 2, 'w': 2.2399999999999998, 'loss': 2.5087999999999995, 'grad': 2.2399999999999998}, {'step': 3, 'w': 1.7919999999999998, 'loss': 1.6056319999999997, 'grad': 1.7919999999999998}, {'step': 4, 'w': 1.4335999999999998, 'loss': 1.0276044799999997, 'grad': 1.4335999999999998}, {'step': 5, 'w': 1.14688, 'loss': 0.6576668671999999, 'grad': 1.14688}, {'step': 6, 'w': 0.9175039999999999, 'loss': 0.4209067950079999, 'grad': 0.9175039999999999}, {'step': 7, 'w': 0.7340031999999999, 'loss': 0.2693803488051199, 'grad': 0.7340031999999999}, {'step': 8, 'w': 0.5872025599999999, 'loss': 0.17240342323527677, 'grad':

In [22]:
# Scaled Dot-Product Attention mechanism
sd.learn("attention")

Visualization(kind='educational_attention', title='Learning: Attention', summary='Scaled Dot-Product Attention: Softmax(Q K^T / sqrt(d_k))', data={'concept': 'attention', 'raw_scores': [[1.0, 0.8, 0.2], [0.66, 0.8, 0.9], [0.2, 0.5, 1.0]], 'scaled_scores': [[0.7071067811865475, 0.565685424949238, 0.1414213562373095], [0.4666904755831214, 0.565685424949238, 0.6363961030678927], [0.1414213562373095, 0.35355339059327373, 0.7071067811865475]], 'weights': [[0.41049316462566465, 0.35635874038575854, 0.23314809498857678], [0.3040422380733266, 0.33568109329263307, 0.36027666863404034], [0.25018981429803355, 0.30931244698641297, 0.44049773871555353]], 'd_k': 2.0}, metadata={}, notes=['Scaling factor: 1/sqrt(d_k) = 1/sqrt(2.0) = 0.707 prevents extreme softmax saturation.', 'Rows of the attention weight matrix represent normalized query-key probabilities summing to 1.0.'], backend='plotly')

## 11. Exporting & Multi-Backend (`.explore()`, `.save()`)

Every SeeDL operation returns a `Visualization` instance supporting:
- `.explore()`: Switch to interactive Plotly mode.
- `.to_figure(backend)`: Return the raw figure object.
- `.save("filename.html" | ".svg" | ".png")`: Export directly to disk.


In [23]:
view = sd.see([10.5, 14.2, 12.8, 16.5, 18.2])

# Switch to Plotly figure
plotly_fig = view.explore()
print("Explored figure type:", type(plotly_fig).__name__)

# Export visualization to self-contained HTML and vector SVG
view.save("examples/showcase_sequence.html")
view.save("examples/showcase_sequence.svg")
print("Exports generated successfully.")

Explored figure type: Figure
Exports generated successfully.
